# Tuấn — MonkeyOCRv2 Phase 1: ICDAR2015 detection evaluation

**Phạm vi:** official-checkpoint evaluation của **DBNet + MonkeyOCRv2-AS / ViTAEv2-S**.
Không training, recognition, TextVQA, tạo degradation hay Phase 2.

Tham chiếu official: **Precision 91,3% · Recall 86,7% · Hmean 88,9%** trên ICDAR2015 test,
với config `dbnet_mkv2vitae_1200e_icdar2015_2gpu_adamw.py` và `HmeanIOUMetric`.
Đây không phải reproduce toàn bộ quá trình huấn luyện.

**Cách dùng trên Kaggle**
1. Import notebook này; bật **Internet** và **GPU T4 ×2**. Baseline dùng GPU 0.
2. Đọc cell cấu hình. Mặc định dùng preparer chính thức **chỉ cho ICDAR2015**.
   Nếu download benchmark bị chặn, attach bản ICDAR2015 đã chuẩn bị theo MMOCR;
   chọn `DATASET_MODE='prepared'` và điền provenance + đường dẫn thật, hoặc để tự tìm JSON.
3. Chạy từ trên xuống. Môi trường Python 3.11 riêng nằm trong `/kaggle/working`;
   notebook kernel không bị thay PyTorch/Paddle. Build MMCV có thể mất khá lâu.
4. Dừng khi có lỗi; xem `logs/` và `blocker.json`. Không đổi metric/threshold để khớp số paper.
5. Sau full evaluation, tải `phase1_monkey_artifacts.zip`; gửi nhóm duyệt trước Phase 2.

**Nguồn đã đối chiếu ngày 30/10/2026:**
- [Hướng dẫn detection, checkpoint, environment và bảng kết quả](https://huggingface.co/HB16888/MonkeyOCRv2_det)
- [MonkeyOCRv2](https://github.com/Yuliang-Liu/MonkeyOCRv2)
- [MMOCR 1.0.1 dataset preparation](https://mmocr.readthedocs.io/en/v1.0.1/user_guides/dataset_prepare.html)
- [MMOCR 1.0.1 testing](https://mmocr.readthedocs.io/en/v1.0.1/user_guides/train_test.html)
- [HmeanIOUMetric](https://mmocr.readthedocs.io/en/stable/basic_concepts/evaluation.html)

Các file source/config chi tiết chưa truy cập được từ máy soạn notebook;
notebook kiểm tra chúng sau khi lấy repo trên Kaggle, không tự thay bằng detector khác.
**Chưa chạy GPU/Kaggle khi bàn giao notebook.**


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, shutil, platform, re, time, textwrap
from datetime import datetime, timezone

ROOT = Path('/kaggle/working/monkey_phase1')
RUN_ID = 'tuan_icdar2015_v1'  # Đổi tên nếu chủ động tạo một run/cấu hình mới.
ART = ROOT / 'artifacts' / RUN_ID
INPUT = Path('/kaggle/input')

MONKEY_URL = 'https://github.com/Yuliang-Liu/MonkeyOCRv2.git'
MONKEY_REF = 'main'  # Lần đầu resolve commit; các lần sau dùng source_lock.json.
DETECTOR_REPO = 'HB16888/MonkeyOCRv2_det'
DETECTOR_FILENAME = 'dbnet_mkv2vitae_icdar2015.pth'
BACKBONE_REPO = 'zenosai/MonkeyOCRv2-AS'
CONFIG_REL = 'configs/textdet/dbnet/dbnet_mkv2vitae_1200e_icdar2015_2gpu_adamw.py'

DATASET_MODE = 'official_preparer'  # 'official_preparer' hoặc 'prepared'
ICDAR2015_ROOT = ''  # prepared: folder chứa textdet_test.json; trống = tìm trong INPUT.
DATASET_PROVENANCE = ''  # prepared: URL/nguồn/version dataset Tuấn đã attach; bắt buộc.
EXPECTED_ANNOTATION_SHA256 = ''  # tùy chọn: hash đã kiểm tra từ nguồn tin cậy.

CUDA_HOME_OVERRIDE = ''  # Trống: tìm nvcc trên Kaggle; không tự cài CUDA toolkit.
ALLOW_CUDA_MINOR_MISMATCH = False  # Chỉ bật nếu chấp nhận build adapted (VD nvcc 12.6/torch cu128).
MAX_BUILD_JOBS = 2  # Hạn chế RAM khi build MMCV; không đặt bằng số GPU.
SMOKE_IMAGES = 20
VIS_IMAGES = 20
NUM_WORKERS = 2
RUN_FULL_EVAL = True
GROUP_TOLERANCE_APPROVED = False  # Chỉ đổi khi nhóm chấp thuận quy ước này.
HMEAN_TOLERANCE_PP = 0.5
REFERENCE = dict(precision=0.913, recall=0.867, hmean=0.889)

assert DATASET_MODE in {'official_preparer', 'prepared'}
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID)
assert 1 <= SMOKE_IMAGES <= 50 and 10 <= VIS_IMAGES <= 20
ART.mkdir(parents=True, exist_ok=True)
for sub in ('logs', 'source', 'scripts'):
    (ART / sub).mkdir(exist_ok=True)
REPO = ROOT / 'MonkeyOCRv2'
ADDON = REPO / 'detection/mmocr'
MMOCR_ROOT = ADDON / 'mmocr'  # Nested checkout do install.sh tạo, không nhầm với ADDON.
VENV = ROOT / 'env_py311'
PY = VENV / 'bin/python'
print('Artifacts:', ART)


## 1. Helpers và preflight
Các lệnh dài ghi log đầy đủ, đồng thời in tiến độ trong notebook.
Lỗi có exit code khác 0 sẽ dừng; không chấm predictions thiếu hoặc dùng metric của smoke test.


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str) + '\n', encoding='utf-8')

def run(cmd, log_name, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    log = ART / 'logs' / log_name
    print('RUN:', subprocess.list2cmdline(cmd), flush=True)
    with log.open('w', encoding='utf-8') as f:
        f.write(json.dumps(dict(cmd=cmd, cwd=str(cwd)), ensure_ascii=False) + '\n')
        proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1)
        for line in proc.stdout:
            f.write(line)
            f.flush()
            print(line, end='', flush=True)
        rc = proc.wait()
    if rc:
        save_json(ART / 'blocker.json', dict(stage=log_name, exit_code=rc, log=str(log),
                                           time=datetime.now(timezone.utc).isoformat()))
        raise RuntimeError(f'BLOCKED: exit {rc}; xem {log}. Không tiếp tục evaluation.')
    return log

def output(cmd, cwd=None, env=None):
    return subprocess.check_output([str(x) for x in cmd], cwd=cwd, env=env, text=True).strip()

def script(name, source):
    p = ART / 'scripts' / name
    compile(source, str(p), 'exec')
    p.write_text(source, encoding='utf-8')
    return p

def block(message):
    save_json(ART / 'blocker.json', dict(message=message, time=datetime.now(timezone.utc).isoformat()))
    raise RuntimeError(message)

assert platform.system() == 'Linux', 'Notebook dành cho Kaggle/Linux GPU; không Run All trên Windows.'
assert shutil.which('git') and shutil.which('nvidia-smi'), 'Bật GPU T4 và Internet trong Kaggle Settings.'
run(['nvidia-smi'], '00_nvidia_smi.log')
run([sys.executable, '-c', 'import sys,platform; print(sys.version); print(platform.platform())'], '00_host.log')

# Tìm nvcc thực, không suy ra toolkit từ CUDA Version trong nvidia-smi.
nvcc_options = []
if CUDA_HOME_OVERRIDE:
    nvcc_options = [Path(CUDA_HOME_OVERRIDE) / 'bin/nvcc']
else:
    nvcc_options = sorted(Path('/usr/local').glob('cuda*/bin/nvcc'))
    if shutil.which('nvcc'):
        nvcc_options.append(Path(shutil.which('nvcc')))
toolkits = {}
for candidate in nvcc_options:
    if candidate.is_file():
        text = output([candidate, '--version'])
        match = re.search(r'release (\d+\.\d+)', text)
        if match:
            toolkits[str(candidate.resolve())] = (match.group(1), candidate.resolve().parent.parent)
print('CUDA toolkits:', toolkits)
preferred = [v for v in toolkits.values() if v[0] == '12.8']
if not preferred and ALLOW_CUDA_MINOR_MISMATCH:
    preferred = sorted([v for v in toolkits.values() if v[0].startswith('12.')], reverse=True)
if not preferred:
    block('Không có nvcc 12.8. Dừng trước khi cài stack lớn. Kiểm tra CUDA_HOME_OVERRIDE; '
          'nếu chỉ có toolkit 12.x khác, có thể bật ALLOW_CUDA_MINOR_MISMATCH để thử môi trường adapted '
          '(có rủi ro build/runtime). H200 cũng phải kiểm tra toolkit, không tự sửa lỗi phiên bản.')
NVCC_VERSION, CUDA_HOME_PATH = preferred[0]
save_json(ART / 'preflight.json', dict(toolkits=toolkits, selected_cuda_home=str(CUDA_HOME_PATH),
                                     nvcc_version=NVCC_VERSION, adapted_cuda=NVCC_VERSION != '12.8'))
print('Selected toolkit:', CUDA_HOME_PATH, NVCC_VERSION)


## 2. Tạo môi trường riêng và khóa source
Theo detection model card: Python 3.11, torch 2.9.0/cu128, torchvision 0.24.0,
MMEngine 0.10.7, MMCV 2.0.1, MMDetection 3.1.0, MMOCR 1.0.1, Transformers 4.57.1.
Đây là stack detection, không cài parser/recognizer hoặc flash-attn từ nhánh khác.

Nếu toolkit không phải 12.8, mặc định dừng. Tùy chọn mismatch chỉ cho cùng major CUDA 12;
log phải ghi môi trường adapted và kiểm tra CUDA ops thực, không coi đó là môi trường giống hệt tác giả.
Source và HF revisions được resolve/lưu ở lần đầu, tái sử dụng lock ở các lần chạy tiếp theo.


In [ ]:
bootstrap = ROOT / 'uv_bootstrap'
if not (bootstrap / 'uv').exists():
    run([sys.executable, '-m', 'pip', 'install', '--target', bootstrap, 'uv'], '01_bootstrap_uv.log')
uv_env = os.environ.copy()
uv_env['PYTHONPATH'] = str(bootstrap)
uv_cmd = [sys.executable, '-m', 'uv']
if not PY.is_file():
    run(uv_cmd + ['python', 'install', '3.11'], '01_python311.log', env=uv_env)
    run(uv_cmd + ['venv', '--python', '3.11', '--seed', VENV], '01_venv.log', env=uv_env)
assert output([PY, '-c', 'import sys; print(f"{sys.version_info.major}.{sys.version_info.minor}")']) == '3.11'

ENV = os.environ.copy()
ENV.pop('PYTHONPATH', None)
ENV.update(PATH=str(VENV / 'bin') + os.pathsep + ENV['PATH'], VIRTUAL_ENV=str(VENV),
           PYTHONPATH=str(MMOCR_ROOT),
           CUDA_HOME=str(CUDA_HOME_PATH), MAX_JOBS=str(MAX_BUILD_JOBS),
           CUDA_VISIBLE_DEVICES='0', MPLBACKEND='Agg', TOKENIZERS_PARALLELISM='false',
           PIP_DISABLE_PIP_VERSION_CHECK='1', PIP_NO_INPUT='1', PIP_NO_CACHE_DIR='1')
# Chỉ checkpoint/code chính thức đã khóa revision/hash; scope trong các subprocess của env này.
# MMEngine cũ gọi torch.load mà không chỉ định weights_only; tương thích với torch >=2.6.
ENV['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

constraints = ART / 'constraints.txt'
constraints.write_text('\n'.join([
    'torch==2.9.0', 'torchvision==0.24.0', 'numpy==1.26.4', 'setuptools<81',
    'mmengine==0.10.7', 'mmcv==2.0.1', 'mmdet==3.1.0', 'mmocr==1.0.1',
    'transformers==4.57.1', 'opencv-python==4.10.0.84',
]) + '\n')
ENV['PIP_CONSTRAINT'] = str(constraints)

lock_path = ART / 'source_lock.json'
locks = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if REPO.exists() and not (REPO / '.git').is_dir():
    block(f'{REPO} tồn tại nhưng không phải Git checkout; không ghi đè.')
if not REPO.exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', '--filter=blob:none', MONKEY_URL, REPO], '02_clone_monkey.log')
target_ref = locks.get('monkey_commit', MONKEY_REF)
run(['git', 'checkout', '--detach', target_ref], '02_checkout_monkey.log', cwd=REPO)
commit = output(['git', 'rev-parse', 'HEAD'], cwd=REPO)
locks['monkey_commit'] = commit
save_json(lock_path, locks)
(ART / 'git_commit.txt').write_text(commit + '\n')
assert (ADDON / 'install.sh').is_file(), 'Repo không chứa detection/mmocr/install.sh; kiểm tra revision.'
shutil.copy2(ADDON / 'install.sh', ART / 'source/install.sh')
print((ADDON / 'install.sh').read_text())


In [ ]:
run([PY, '-m', 'pip', 'install', 'torch==2.9.0', 'torchvision==0.24.0',
     '--index-url', 'https://download.pytorch.org/whl/cu128'], '03_torch.log', env=ENV)
run([PY, '-m', 'pip', 'install', 'numpy==1.26.4', 'setuptools<81', 'wheel', 'ninja',
     'mmengine==0.10.7', 'opencv-python==4.10.0.84'], '03_build_dependencies.log', env=ENV)
gpu_probe = script('gpu_probe.py', """
import json, torch
assert torch.cuda.is_available(), 'Torch không sử dụng được GPU; kiểm tra driver/runtime.'
x = torch.ones((8, 8), device='cuda')
assert (x @ x).sum().item() == 512
gpus = [dict(name=torch.cuda.get_device_name(i), capability=torch.cuda.get_device_capability(i))
        for i in range(torch.cuda.device_count())]
print(json.dumps(dict(torch=torch.__version__, cuda=torch.version.cuda, gpus=gpus)))
print('ARCH_LIST=' + ';'.join(sorted({'.'.join(map(str, torch.cuda.get_device_capability(i)))
                                     for i in range(torch.cuda.device_count())})))
""")
probe_log = run([PY, gpu_probe], '03_gpu_probe.log', env=ENV)
arch_match = re.search(r'ARCH_LIST=(.+)', probe_log.read_text())
assert arch_match
ENV['TORCH_CUDA_ARCH_LIST'] = arch_match.group(1).strip()
print('Build CUDA architectures:', ENV['TORCH_CUDA_ARCH_LIST'])

MMCV_SRC = ROOT / 'mmcv_src'
if not MMCV_SRC.exists():
    run(['git', 'clone', '--branch', 'v2.0.1', '--depth', '1',
         'https://github.com/open-mmlab/mmcv.git', MMCV_SRC], '04_clone_mmcv.log', env=ENV)
assert output(['git', 'describe', '--tags', '--exact-match'], cwd=MMCV_SRC) == 'v2.0.1'
setup = MMCV_SRC / 'setup.py'
original_setup = output(['git', 'show', 'HEAD:setup.py'], cwd=MMCV_SRC)
setup.write_text(original_setup.replace('-std=c++14', '-std=c++17'))
run(['git', 'diff'], '04_mmcv_cxx17.patch', cwd=MMCV_SRC)
build_env = dict(ENV, MMCV_WITH_OPS='1', FORCE_CUDA='1')
run([PY, '-m', 'pip', 'install', '-v', '-e', MMCV_SRC, '--no-build-isolation'],
    '04_build_mmcv.log', env=build_env)
locks['mmcv_commit'] = output(['git', 'rev-parse', 'HEAD'], cwd=MMCV_SRC)
save_json(lock_path, locks)


In [ ]:
# Official installer tạo nested ./mmocr và copy configs/patches. Chạy một lần mỗi source commit.
installed_marker = ADDON / '.phase1_installed_commit'
if not installed_marker.exists():
    if MMOCR_ROOT.exists():
        block('Nested MMOCR đã tồn tại nhưng chưa có marker. Xem logs cài trước; không tự overwrite. '
              'Dùng Kaggle session sạch để cài lại nếu installer trước bị gián đoạn.')
    run(['bash', 'install.sh'], '05_official_install.log', cwd=ADDON, env=ENV)
    assert (MMOCR_ROOT / 'tools/test.py').is_file()
    installed_marker.write_text(commit)
else:
    assert installed_marker.read_text().strip() == commit, 'Source khác marker; cần session/source root mới.'
assert (MMOCR_ROOT / '.git').is_dir()
locks['mmocr_commit'] = output(['git', 'rev-parse', 'HEAD'], cwd=MMOCR_ROOT)
save_json(lock_path, locks)

run([PY, '-m', 'pip', 'install', 'mmdet==3.1.0', 'transformers==4.57.1',
     'huggingface_hub', 'safetensors', 'einops', 'timm', 'Pillow', 'scipy',
     'shapely', 'pyclipper', 'imgaug', 'ftfy', 'lmdb'], '05_detection_dependencies.log', env=ENV)
run([PY, '-m', 'pip', 'install', '-e', MMOCR_ROOT], '05_mmocr_editable.log', env=ENV)
run(['git', 'diff', '--stat'], '05_mmocr_patch_stat.log', cwd=MMOCR_ROOT)
run(['git', 'diff'], '05_mmocr.patch', cwd=MMOCR_ROOT)

ops_probe = script('ops_probe.py', """
import importlib.metadata as im, json
import torch, mmcv, mmengine, mmdet, mmocr, transformers
from mmcv.ops import nms, ModulatedDeformConv2d
expected = {'torch':'2.9.0', 'torchvision':'0.24.0', 'mmcv':'2.0.1', 'mmengine':'0.10.7',
            'mmdet':'3.1.0', 'mmocr':'1.0.1', 'transformers':'4.57.1', 'numpy':'1.26.4'}
actual = {k: im.version(k) for k in expected}
for k, v in expected.items():
    assert actual[k].split('+')[0] == v, (k, actual[k], v)
boxes = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]], device='cuda')
dets, keep = nms(boxes, torch.tensor([.9,.8], device='cuda'), .5)
assert keep.numel() == 1
op = ModulatedDeformConv2d(2, 2, 3, padding=1).cuda()
y = op(torch.randn(1,2,8,8,device='cuda'), torch.zeros(1,18,8,8,device='cuda'),
       torch.ones(1,9,8,8,device='cuda'))
assert y.shape == (1,2,8,8) and torch.isfinite(y).all()
torch.cuda.synchronize()
print(json.dumps(actual, indent=2)); print('CUDA nms + modulated deform conv: OK')
""")
run([PY, ops_probe], '06_imports_cuda_ops.log', cwd=MMOCR_ROOT, env=ENV)
run([PY, '-m', 'pip', 'freeze'], 'env_versions.txt', env=ENV)
shutil.copy2(ART / 'logs/env_versions.txt', ART / 'env_versions.txt')


## 3. Checkpoint detection và backbone đúng revision
Chỉ tải checkpoint ICDAR2015 và encoder AS cần thiết. Không tải parser, training data pretraining,
hoặc checkpoint Total-Text/CTW1500. Ghi SHA256 + HF revision; không tự chọn file gần giống khi thiếu file.


In [ ]:
download_script = script('download_artifacts.py', """
import hashlib, json, sys
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
root, lock_path, det_repo, det_file, backbone_repo = sys.argv[1:]
root, lock_path = Path(root), Path(lock_path)
locks = json.loads(lock_path.read_text())
api = HfApi()
items = [(det_repo, 'detector_hf_revision'), (backbone_repo, 'backbone_hf_revision')]
for repo, key in items:
    if key not in locks:
        locks[key] = api.model_info(repo).sha
        lock_path.write_text(json.dumps(locks, indent=2))
files = api.list_repo_files(det_repo, revision=locks['detector_hf_revision'])
assert det_file in files, f'Không có {det_file}. Các pth có thật: {[x for x in files if x.endswith(".pth")]}'
det = hf_hub_download(det_repo, det_file, revision=locks['detector_hf_revision'],
                      local_dir=root/'model_weight')
snapshot_download(backbone_repo, revision=locks['backbone_hf_revision'],
                  local_dir=root/'pretrained/monkeyocrv2_as')
assert (root/'pretrained/monkeyocrv2_as/config.json').is_file()
print('Detector:', det)
""")
run([PY, download_script, MMOCR_ROOT, lock_path, DETECTOR_REPO, DETECTOR_FILENAME, BACKBONE_REPO],
    '07_download_artifacts.log', env=ENV)
locks = json.loads(lock_path.read_text())
CHECKPOINT = MMOCR_ROOT / 'model_weight' / DETECTOR_FILENAME
CONFIG = MMOCR_ROOT / CONFIG_REL
assert CHECKPOINT.is_file() and CONFIG.is_file()
shutil.copy2(CONFIG, ART / 'source' / CONFIG.name)
hashes = dict(detector=dict(path=str(CHECKPOINT), sha256=sha256_file(CHECKPOINT),
                           bytes=CHECKPOINT.stat().st_size,
                           url=f'https://huggingface.co/{DETECTOR_REPO}/resolve/{locks["detector_hf_revision"]}/{DETECTOR_FILENAME}'),
              config=dict(path=str(CONFIG), sha256=sha256_file(CONFIG)), backbone={})
for p in sorted((MMOCR_ROOT / 'pretrained/monkeyocrv2_as').rglob('*')):
    if p.is_file() and '.cache' not in p.parts:
        hashes['backbone'][str(p.relative_to(MMOCR_ROOT))] = dict(sha256=sha256_file(p), bytes=p.stat().st_size)
save_json(ART / 'artifact_hashes.json', hashes)
(ART / 'checkpoint_sha256.txt').write_text(hashes['detector']['sha256'] + '  ' + DETECTOR_FILENAME + '\n')
print(json.dumps(hashes['detector'], indent=2))


## 4. ICDAR2015 test — chuẩn bị và audit
Default gọi `prepare_dataset.py icdar2015 --task textdet` của repo đã patch, không gọi prepare-all.
Nếu download cần đăng nhập hoặc link hỏng: cell dừng, attach dataset hợp lệ rồi dùng `prepared`.
Không thay bằng TextVQA hoặc một bộ ảnh khác. Đường dẫn Kaggle chỉ được chọn từ file có thật.

Audit kiểm tra đủ 500 ảnh test, schema MMOCR, ảnh đọc được, kích thước, polygons,
ignore flags và hash từng ảnh/annotation. Số lượng đúng **chưa tự chứng minh nguồn benchmark**;
với bản attach cần ghi nguồn/version để nhóm review.


In [ ]:
if DATASET_MODE == 'official_preparer':
    DATA_ROOT = MMOCR_ROOT / 'data/icdar2015'
    if not (DATA_ROOT / 'textdet_test.json').is_file():
        prepare_tool = MMOCR_ROOT / 'tools/dataset_converters/prepare_dataset.py'
        assert prepare_tool.is_file()
        run([PY, prepare_tool, 'icdar2015', '--task', 'textdet'],
            '08_prepare_icdar2015.log', cwd=MMOCR_ROOT, env=ENV)
    provenance = dict(mode=DATASET_MODE, source='Official patched MMOCR ICDAR2015 preparer',
                      dataset_zoo_commit=locks['mmocr_commit'])
    zoo = MMOCR_ROOT / 'dataset_zoo/icdar2015'
    if zoo.is_dir():
        shutil.copytree(zoo, ART / 'source/icdar2015_dataset_zoo', dirs_exist_ok=True)
else:
    if not DATASET_PROVENANCE.strip():
        block('Điền DATASET_PROVENANCE cho bản ICDAR2015 attach; không suy ra nguồn từ số lượng ảnh.')
    if ICDAR2015_ROOT:
        DATA_ROOT = Path(ICDAR2015_ROOT)
    else:
        candidates = sorted(INPUT.rglob('textdet_test.json'))
        print('Prepared annotation candidates:', [str(p) for p in candidates])
        if len(candidates) != 1:
            block('Cần đúng một annotation candidate, hoặc điền ICDAR2015_ROOT từ danh sách thực.')
        DATA_ROOT = candidates[0].parent
    provenance = dict(mode=DATASET_MODE, source=DATASET_PROVENANCE)
assert (DATA_ROOT / 'textdet_test.json').is_file()
save_json(ART / 'dataset_provenance.json', provenance)

audit_source = r"""
import hashlib, json, math, re, sys
from pathlib import Path
from PIL import Image

def sha(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def audit(root, expected_sha=''):
    root=Path(root).resolve(); ann=root/'textdet_test.json'
    digest=sha(ann)
    if expected_sha: assert digest==expected_sha, 'Annotation SHA mismatch'
    obj=json.loads(ann.read_text(encoding='utf-8-sig'))
    assert isinstance(obj.get('metainfo'),dict) and isinstance(obj.get('data_list'),list), 'Sai MMOCR schema'
    rows=obj['data_list']; assert len(rows)==500, f'ICDAR2015 test phải có 500 ảnh, nhận {len(rows)}'
    records=[]; names=set(); valid=ignored=0
    for r in rows:
        assert {'img_path','height','width','instances'} <= r.keys()
        name=Path(r['img_path']).name
        assert name not in names, f'Ảnh lặp: {name}'
        names.add(name)
        p=root/r['img_path']
        assert p.is_file(), f'img_path phải resolve tương đối từ DATA_ROOT: {p}'
        assert p.resolve().is_relative_to(root), 'Đường dẫn ảnh nằm ngoài dataset root'
        with Image.open(p) as im:
            assert im.size==(r['width'],r['height']), (p, im.size, r['width'],r['height'])
            im.convert('RGB').load()
        assert isinstance(r['instances'],list)
        for inst in r['instances']:
            points=inst['polygon']
            assert len(points)>=8 and len(points)%2==0 and all(math.isfinite(float(x)) for x in points)
            assert 'ignore' in inst and inst['ignore'] in (True,False,0,1), 'Thiếu ignore flag'
            ignored+=int(bool(inst['ignore'])); valid+=int(not inst['ignore'])
        records.append(dict(img_path=r['img_path'],resolved_path=str(p.resolve()),sha256=sha(p),
                            width=r['width'],height=r['height']))
    expected={f'img_{i}.jpg' for i in range(1,501)}
    assert names==expected, 'Tên ảnh không khớp ICDAR2015 test img_1.jpg...img_500.jpg; kiểm tra benchmark'
    assert valid>0 and ignored>0, 'GT có dấu hiệu thiếu valid/ignored regions'
    return dict(annotation_sha256=digest, annotation_path=str(ann), num_images=len(rows),
                valid_gt=valid, ignored_gt=ignored, records=records, metainfo=obj['metainfo'])

if __name__=='__main__':
    result=audit(sys.argv[1],sys.argv[3] if len(sys.argv)>3 else '')
    Path(sys.argv[2]).write_text(json.dumps(result,indent=2))
    print(json.dumps({k:v for k,v in result.items() if k!='records'},indent=2))
"""
audit_script = script('audit_icdar2015.py', audit_source)
run([PY, audit_script, DATA_ROOT, ART / 'dataset_audit.json', EXPECTED_ANNOTATION_SHA256],
    '08_dataset_audit.log', env=ENV)
dataset_audit = json.loads((ART / 'dataset_audit.json').read_text())


## 5. Instrumentation: lưu kết quả, không đổi evaluator
Hook chỉ đọc predictions/metrics và lưu tối đa 20 overlays: **xanh = GT valid, xám = ignored,
đỏ = predictions**. Vẽ mọi prediction được detector trả ra; không thêm ngưỡng vào evaluation.
Smoke 20 ảnh tách khỏi full 500 ảnh, không dùng smoke metric để nhận reproduce thành công.


In [ ]:
hook_source = r"""
import json, time
from pathlib import Path
import cv2, numpy as np
from mmengine.hooks import Hook
from mmengine.registry import HOOKS

def serial(x):
    if hasattr(x,'detach'): return x.detach().cpu().tolist()
    if isinstance(x,np.ndarray): return x.tolist()
    if isinstance(x,dict): return {str(k):serial(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)): return [serial(v) for v in x]
    if isinstance(x,np.generic): return x.item()
    return x

@HOOKS.register_module()
class Phase1AuditHook(Hook):
    def __init__(self, output_dir, vis_limit=20):
        self.out=Path(output_dir); self.out.mkdir(parents=True,exist_ok=True)
        self.vis_limit=vis_limit; self.seen=set(); self.count=0
    def before_test(self,runner):
        self.start=time.perf_counter()
        self.f=(self.out/'predictions.jsonl').open('w',encoding='utf-8')
        (self.out/'visual_examples').mkdir(exist_ok=True)
    def after_test_iter(self,runner,batch_idx,data_batch=None,outputs=None):
        for sample in outputs:
            path=str(Path(sample.metainfo['img_path']).resolve())
            assert path not in self.seen, f'Duplicate inference image: {path}'
            self.seen.add(path)
            pred=sample.pred_instances
            polys=serial(pred.polygons); scores=serial(pred.scores)
            assert len(polys)==len(scores)
            self.f.write(json.dumps(dict(image_path=path,polygons=polys,scores=scores))+'\n')
            self.f.flush()
            if self.count<self.vis_limit:
                im=cv2.imread(path); assert im is not None
                gt=sample.gt_instances
                for poly,ignore in zip(serial(gt.polygons),serial(gt.ignored)):
                    cv2.polylines(im,[np.asarray(poly).reshape(-1,2).astype(np.int32)],True,
                                  (150,150,150) if ignore else (0,200,0),2)
                for poly in polys:
                    cv2.polylines(im,[np.asarray(poly).reshape(-1,2).astype(np.int32)],True,(0,0,255),2)
                assert cv2.imwrite(str(self.out/'visual_examples'/Path(path).name),im)
            self.count+=1
    def after_test_epoch(self,runner,metrics=None):
        self.f.close()
        record=dict(metrics=serial(metrics),processed_count=self.count,
                    image_paths=sorted(self.seen),wall_seconds_with_export=time.perf_counter()-self.start)
        (self.out/'official_metrics_raw.json').write_text(json.dumps(record,indent=2))
"""
HOOK_FILE = MMOCR_ROOT / 'phase1_audit_hook.py'
compile(hook_source, str(HOOK_FILE), 'exec')
HOOK_FILE.write_text(hook_source)
shutil.copy2(HOOK_FILE, ART / 'source/phase1_audit_hook.py')


In [ ]:
config_builder_source = r"""
import copy, inspect, json, sys
from pathlib import Path
from mmengine.config import Config
from mmengine.utils import import_modules_from_strings

source, data_root, artifact_dir, smoke_n, workers, vis_n = sys.argv[1:]
data_root=Path(data_root).resolve(); out=Path(artifact_dir).resolve()
cfg=Config.fromfile(source)
original=copy.deepcopy(cfg)
cfg.dump(str(out/'source/official_resolved_config.py'))
text=repr(cfg.model).lower()
assert 'dbnet' in text and any(x in text for x in ('mkv2','monkey','vitae')), 'Sai detector/backbone'
evaluator=cfg.test_evaluator
items=evaluator if isinstance(evaluator,list) else [evaluator]
assert len(items)==1 and items[0]['type'].split('.')[-1]=='HmeanIOUMetric', evaluator
ds=cfg.test_dataloader.dataset
assert ds['type'].split('.')[-1]=='OCRDataset', 'Chưa hỗ trợ wrapper dataset; không tự sửa protocol'
assert ds.get('indices') is None, 'Official test dataset đang bị subset'
ds['data_root']=str(data_root)
ds['ann_file']='textdet_test.json'
# MMOCR prepared JSON img_path đã chứa textdet_imgs/test; không thêm prefix lần nữa.
ds['data_prefix']=dict(img_path='')
cfg.test_dataloader.batch_size=1
cfg.test_dataloader.num_workers=int(workers)
cfg.test_dataloader.persistent_workers=int(workers)>0
custom=cfg.get('custom_imports',dict(imports=[],allow_failed_imports=False))
imports=custom.get('imports',[])
if isinstance(imports,str): imports=[imports]
cfg.custom_imports=dict(imports=list(imports)+['phase1_audit_hook'],allow_failed_imports=False)
import_modules_from_strings(**cfg.custom_imports)
from mmocr.utils import register_all_modules
register_all_modules(init_default_scope=True)
from mmocr.registry import METRICS
metric=METRICS.build(copy.deepcopy(items[0]))
protocol=dict(config_test_evaluator=evaluator,actual_strategy=metric.strategy,
              match_iou_thr=metric.match_iou_thr,ignore_precision_thr=metric.ignore_precision_thr,
              pred_score_thrs=metric.pred_score_thrs.tolist())
(out/'evaluator_protocol.json').write_text(json.dumps(protocol,indent=2))
existing_hooks=list(cfg.get('custom_hooks',[]))
for mode in ('smoke','full'):
    c=copy.deepcopy(cfg)
    if mode=='smoke': c.test_dataloader.dataset.indices=list(range(int(smoke_n)))
    c.work_dir=str(out/mode/'work_dir')
    c.custom_hooks=existing_hooks+[dict(type='Phase1AuditHook',output_dir=str(out/mode),vis_limit=int(vis_n))]
    assert c.model==original.model, 'Không được sửa model/postprocessing'
    assert c.test_evaluator==original.test_evaluator, 'Không được sửa evaluator/thresholds'
    assert c.test_dataloader.dataset.pipeline==original.test_dataloader.dataset.pipeline, 'Không được sửa pipeline'
    c.dump(str(out/f'{mode}_config.py'))
changes=dict(data_root=str(data_root),ann_file='textdet_test.json',data_prefix={'img_path':''},
             batch_size=1,num_workers=int(workers),instrumentation='Phase1AuditHook',
             smoke_indices=f'first {smoke_n}; FULL has no subset')
(out/'config_overrides.json').write_text(json.dumps(changes,indent=2))
print(json.dumps(protocol,indent=2))
"""
config_builder = script('build_eval_configs.py', config_builder_source)
run([PY, config_builder, CONFIG, DATA_ROOT, ART, SMOKE_IMAGES, NUM_WORKERS, VIS_IMAGES],
    '09_build_configs.log', cwd=MMOCR_ROOT, env=ENV)
print((ART / 'evaluator_protocol.json').read_text())


**Điểm cần phân biệt trong handoff:** `HmeanIOUMetric` có thể tự sweep score thresholds theo cấu hình
official. Phase 1 giữ nguyên hành vi này để so sánh đúng bảng tác giả; lưu toàn bộ protocol/log.
Không thay range sau khi xem kết quả. Phase 2 cần protocol chung được chốt riêng và không dùng
test results để chỉnh threshold. Notebook này không triển khai Phase 2.


In [ ]:
strict_check = script('strict_checkpoint_check.py', """
import json, sys
from mmengine.config import Config
from mmengine.utils import import_modules_from_strings
from mmengine.runner import load_checkpoint
from mmocr.utils import register_all_modules
from mmocr.registry import MODELS
cfg=Config.fromfile(sys.argv[1])
if cfg.get('custom_imports'): import_modules_from_strings(**cfg.custom_imports)
register_all_modules(init_default_scope=True)
model=MODELS.build(cfg.model)
# Full detector checkpoint phải tương thích; không chấp nhận fallback sang random detection head.
ckpt=load_checkpoint(model,sys.argv[2],map_location='cpu',strict=True)
print('STRICT CHECKPOINT LOAD: OK')
meta=ckpt.get('meta',{}) if isinstance(ckpt,dict) else {}
print('Checkpoint meta keys:', sorted(meta))
""")
run([PY, strict_check, ART / 'full_config.py', CHECKPOINT],
    '09_strict_checkpoint_load.log', cwd=MMOCR_ROOT, env=ENV)

test_help = output([PY, 'tools/test.py', '--help'], cwd=MMOCR_ROOT, env=ENV)
(ART / 'source/test_help.txt').write_text(test_help)
assert '--work-dir' in test_help
print('Checkpoint/config preflight hoàn tất; bắt đầu smoke test, không phải kết quả paper.')
run([PY, 'tools/test.py', ART / 'smoke_config.py', CHECKPOINT,
     '--work-dir', ART / 'smoke/work_dir'], '10_smoke_eval.log', cwd=MMOCR_ROOT, env=ENV)
smoke = json.loads((ART / 'smoke/official_metrics_raw.json').read_text())
assert smoke['processed_count'] == SMOKE_IMAGES
assert len((ART / 'smoke/predictions.jsonl').read_text().splitlines()) == SMOKE_IMAGES
print('SMOKE ONLY:', smoke['metrics'])


## 6. Full official evaluation — single GPU
Chạy đúng 500 ảnh, cùng checkpoint/config gốc. Hook xuất predictions và metrics từ runner;
không tự tính evaluator thay thế. Không bật AMP/TTA hoặc sửa resize/postprocessing.
Nếu lỗi OOM, không tự hạ input resolution; ghi blocker và xử lý tài nguyên trước.


In [ ]:
if RUN_FULL_EVAL:
    full_log = run([PY, 'tools/test.py', ART / 'full_config.py', CHECKPOINT,
                    '--work-dir', ART / 'full/work_dir'], 'official_eval.log', cwd=MMOCR_ROOT, env=ENV)
    shutil.copy2(full_log, ART / 'official_eval.log')
else:
    print('RUN_FULL_EVAL=False: chỉ có smoke; chưa có kết quả reproduction.')


In [ ]:
def normalize_metrics(raw):
    metrics = raw['metrics']
    result = {}
    for name in ('precision', 'recall', 'hmean'):
        values = [float(v) for k, v in metrics.items() if k.split('/')[-1] == name]
        assert len(values) == 1, f'Metric {name} thiếu hoặc mơ hồ: {metrics}'
        assert 0 <= values[0] <= 1
        result[name] = values[0]
    return result

if RUN_FULL_EVAL:
    raw = json.loads((ART / 'full/official_metrics_raw.json').read_text())
    assert raw['processed_count'] == dataset_audit['num_images'] == 500
    expected_paths = {r['resolved_path'] for r in dataset_audit['records']}
    assert set(raw['image_paths']) == expected_paths and len(raw['image_paths']) == 500
    predictions = [json.loads(s) for s in (ART / 'full/predictions.jsonl').read_text().splitlines()]
    assert len(predictions) == 500 and {p['image_path'] for p in predictions} == expected_paths
    assert sha256_file(CHECKPOINT) == hashes['detector']['sha256']
    measured = normalize_metrics(raw)
    delta = {k: 100 * (measured[k] - REFERENCE[k]) for k in REFERENCE}
    within = abs(delta['hmean']) <= HMEAN_TOLERANCE_PP
    result = dict(evaluation_type='official_checkpoint_evaluation_reproduction',
                  benchmark='ICDAR2015 test',num_images=500,measured=measured,reference=REFERENCE,
                  delta_percentage_points=delta,numerically_within_proposed_hmean_tolerance=within,
                  proposed_tolerance_pp=HMEAN_TOLERANCE_PP,group_tolerance_approved=GROUP_TOLERANCE_APPROVED,
                  status='EVALUATED_REQUIRES_GROUP_REVIEW' if within else 'REVIEW_MISMATCH',
                  phase2_authorized=False,source_lock=locks,dataset=provenance,
                  nvcc_version=NVCC_VERSION,adapted_cuda=NVCC_VERSION != '12.8',
                  wall_seconds_includes_export=raw['wall_seconds_with_export'])
    save_json(ART / 'official_metrics.json', result)
    print('Metric        measured(%)   reference(%)   delta(pp)')
    for name in REFERENCE:
        print(f'{name:12s} {100*measured[name]:10.3f} {100*REFERENCE[name]:14.3f} {delta[name]:+11.3f}')
    print('Status:', result['status'], '| Phase 2: cần nhóm phê duyệt riêng.')


## 7. Xem overlay và đóng gói bàn giao
Kiểm tra 10–20 ví dụ, đủ benchmark, checkpoint load nghiêm ngặt, và evaluator trước khi kết luận.
Sai lệch ≤0,5 điểm phần trăm Hmean là đề xuất nội bộ trong handoff, không phải tiêu chí paper.
Ngay cả khi số gần nhau, notebook không tự phê duyệt Phase 2.

Nếu debug, giữ lại log cũ bằng run ID mới; không tune trên test rồi chỉ báo lần tốt nhất.


In [ ]:
from IPython.display import display, Image as DisplayImage, FileLink
vis_dir = ART / ('full' if RUN_FULL_EVAL else 'smoke') / 'visual_examples'
examples = sorted(vis_dir.glob('*.jpg'))
for p in examples[:4]:
    print(p.name)
    display(DisplayImage(filename=str(p), width=850))
print('Saved overlays:', len(examples), 'in', vis_dir)

config_inventory = {}
for p in sorted(ART.rglob('*')):
    if p.is_file():
        config_inventory[str(p.relative_to(ART))] = dict(bytes=p.stat().st_size,sha256=sha256_file(p))
save_json(ART / 'inventory_sha256.json', config_inventory)
readme = f"""Tuấn — MonkeyOCRv2 Phase 1, run {RUN_ID}
Scope: DBNet + MonkeyOCRv2-AS, official-checkpoint evaluation on ICDAR2015 test.
Reference P/R/Hmean (%): 91.3 / 86.7 / 88.9.
Read official_metrics.json only for FULL evaluation; smoke metrics are not reproduction results.
See source_lock.json, artifact_hashes.json, evaluator_protocol.json, config_overrides.json,
dataset_provenance.json, dataset_audit.json, env_versions.txt, source/, logs/.
Full predictions: full/predictions.jsonl. Overlays: full/visual_examples/.
No training, TextVQA benchmark, threshold tuning, restoration or Phase 2 performed here.
Group review and separate approval are required for Phase 2.
"""
(ART / 'README.txt').write_text(readme,encoding='utf-8')
archive = shutil.make_archive(str(ROOT / 'phase1_monkey_artifacts'), 'zip', ART)
print('DOWNLOAD:', archive)
display(FileLink(archive))


## Khi có blocker

| Triệu chứng | Kiểm tra / xử lý |
|---|---|
| Thiếu nvcc 12.8 | Xem preflight; chỉ thử CUDA 12.x khác khi chấp nhận adapted environment. Không đổi model. |
| Build MMCV bị killed | Giảm `MAX_BUILD_JOBS` về 1; giữ logs; kiểm tra RAM/disk. |
| `undefined symbol` / CUDA ops lỗi | So compiler, torch CUDA và toolkit; không thay bằng mmcv-lite. |
| Dataset download 403/link hỏng | Attach ICDAR2015 đã chuẩn bị, điền provenance và `DATASET_MODE='prepared'`. |
| Thiếu config/checkpoint | Đối chiếu repo/HF revision; không tự dùng PSENet hay parser thay DBNet. |
| Strict load lỗi | Dừng, kiểm tra đúng file/config/checkpoint; không bỏ `strict=True` để chạy tiếp. |
| Full Hmean lệch | Kiểm tra dataset, ignore flags, resolved config, resize, evaluator, source/checkpoint hashes. |
| Smoke thành công | Chỉ xác nhận pipeline hoạt động; vẫn phải chạy full test và nhóm review. |

Nếu lỗi giữa chừng: tải riêng `logs/`, `blocker.json` và các JSON/config đã có trong tab Output;
cell đóng gói cuối chưa chạy thì chưa có ZIP. Giữ notebook output khi Save Version để audit.
